<a href="https://colab.research.google.com/github/xmessi08/ML-Project6/blob/main/ML_Project6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import re

# Импортируем стандартный датасет текстов из sklearn (темы: медицина, космос, компьютеры и т.д.)
from sklearn.datasets import fetch_20newsgroups

# Инструменты предобработки и векторизации
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Модель классификации
from sklearn.naive_bayes import MultinomialNB

# Метрики качества
from sklearn.metrics import accuracy_score, classification_report, f1_score

In [2]:
# Выбираем две категории для простоты и наглядности классификации
categories = ['comp.graphics', 'sci.space']

print("Загружаем датасет...")
data = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))

# Создаем DataFrame для удобства
df = pd.DataFrame({'text': data.data, 'target': data.target})

# Уберем пустые тексты, если они есть
df = df[df['text'].str.strip() != ''].reset_index(drop=True)

print(f"Всего документов в датасете: {len(df)}")
print(df.head(3))

Загружаем датасет...
Всего документов в датасете: 1910
                                                text  target
0  \nActually, Hiten wasn't originally intended t...       1
1  Archive-name: space/launchers\nLast-modified: ...       1
2  \nNeil Gehrels is Prof. Tom Gehrels son.  Tom ...       1


In [6]:
def clean_text(text):
    # Приведение к нижнему регистру
    text = text.lower()
    # Удаление HTML-тегов (если есть)
    text = re.sub(r'<.*?>', '', text)
    # Удаление цифр
    text = re.sub(r'\d+', '', text)
    # Удаление пунктуации и спецсимволов (оставляем только буквы и пробелы)
    text = re.sub(r'[^\w\s]', ' ', text)
    # Удаление лишних пробелов
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Применяем функцию очистки ко всем текстам
df['clean_text'] = df['text'].apply(clean_text)

# Посмотрим на результат до и после
print("ДО:\n", df['text'].iloc[0][:200])
print("\nПОСЛЕ:\n", df['clean_text'].iloc[0][:200])

ДО:
 
Actually, Hiten wasn't originally intended to go into lunar orbit at all,
so it indeed didn't have much fuel on hand.  The lunar-orbit mission was
an afterthought, after Hagoromo (a tiny subsatellite

ПОСЛЕ:
 actually hiten wasn t originally intended to go into lunar orbit at all so it indeed didn t have much fuel on hand the lunar orbit mission was an afterthought after hagoromo a tiny subsatellite deploy


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'],
    df['target'],
    test_size=0.2,
    random_state=42,
    stratify=df['target']
)

print( размер обучающей выборки: {len(X_train)})
print( размер тестовой выборки: {len(X_test)})

размер обучающей выборки: 1528
размер тестовой выборки: 382


In [9]:
# Инициализируем TF-IDF векторизатор с ограничением словаря (min_df и max_df)
vectorizer = TfidfVectorizer(min_df=3, max_df=0.9, stop_words='english')

# Обучаем векторизатор ТОЛЬКО на train и преобразуем train и test
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print("Размерность полученного признакового пространства (матрицы train):", X_train_vec.shape)

Размерность полученного признакового пространства (матрицы train): (1528, 7082)


In [10]:
# Обучаем наивный байесовский классификатор
model = MultinomialNB()
model.fit(X_train_vec, y_train)

# Предсказываем на тестовой выборке
y_pred = model.predict(X_test_vec)

# Считаем метрики
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')

print(f"Accuracy: {acc:.4f}")
print(f"F1-score: {f1:.4f}")
print("\nПодробный отчет о классификации:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9450
F1-score: 0.9450

Подробный отчет о классификации:
              precision    recall  f1-score   support

           0       0.94      0.95      0.95       191
           1       0.95      0.94      0.94       191

    accuracy                           0.95       382
   macro avg       0.95      0.95      0.95       382
weighted avg       0.95      0.95      0.95       382



In [11]:
# Соберем результаты теста в один DataFrame для поиска ошибок
test_results = pd.DataFrame({
    'text': X_test.values,
    'true_label': y_test.values,
    'pred_label': y_pred
})

# Находим индексы ошибок
errors_idx = test_results[test_results['true_label'] != test_results['pred_label']].index

print(f"Всего ошибок на тесте: {len(errors_idx)}")
print("\nПримеры ошибок для анализа:")

# Выведем первые 5 ошибок
for i, idx in enumerate(errors_idx[:5]):
    print(f"\n--- Ошибка № {i+1} ---")
    print(f"Текст (очищенный): {test_results.loc[idx, 'text'][:300]}...")
    print(f"Истинный класс: {test_results.loc[idx, 'true_label']} | Предсказанный класс: {test_results.loc[idx, 'pred_label']}")

Всего ошибок на тесте: 21

Примеры ошибок для анализа:

--- Ошибка № 1 ---
Текст (очищенный): invitation to the th scia the th scandinavian conference on image analysis will be arranged by the norwegian society for image processing and pattern recognition nobim and sponsored by the international association for pattern recognition iapr the conference will be held in tromsoe from th th may tr...
Истинный класс: 0 | Предсказанный класс: 1

--- Ошибка № 2 ---
Текст (очищенный): the nasm photo archives are open to the public all or almost all still pictures in the collection are available for viewing but i don t know about films at least it might be worth a try i m not sure if appointments are necessary but i think not good luck and let us know what you find...
Истинный класс: 1 | Предсказанный класс: 0

--- Ошибка № 3 ---
Текст (очищенный): hello all i apologize i haven t published my astro ftp list since march now i haven t tested all the sites included into the list i would notified all

In [19]:
from sklearn.feature_extraction.text import CountVectorizer

# 1. Векторизация через CountVectorizer
count_vectorizer = CountVectorizer(min_df=3, max_df=0.9, stop_words='english')
X_train_cv = count_vectorizer.fit_transform(X_train)
X_test_cv = count_vectorizer.transform(X_test)

# 2. Обучение модели на CountVectorizer
model_cv = MultinomialNB()
model_cv.fit(X_train_cv, y_train)
y_pred_cv = model_cv.predict(X_test_cv)

# 3. Расчет метрик для CountVectorizer
acc_cv = accuracy_score(y_test, y_pred_cv)
f1_cv = f1_score(y_test, y_pred_cv, average='weighted')

# 4. Сравнение результатов в таблице
comparison_df = pd.DataFrame({
    'Метод векторизации': ['CountVectorizer (BoW)', 'TfidfVectorizer (TF-IDF)'],
    'Accuracy': [acc_cv, acc],
    'F1-score': [f1_cv, f]
})

print("Результаты сравнения:")
print(comparison_df.to_string(index=False))

Результаты сравнения:
      Метод векторизации  Accuracy  F1-score
   CountVectorizer (BoW)  0.937173  0.937171
TfidfVectorizer (TF-IDF)  0.945026  0.945023
